# Phase 5 — Hyperparameter Optimisation

Base model: **XGBoost Phase 4** (F1 Macro=0.8605, selected features).  
Target: F1 Macro ≥ 0.90 via RandomizedSearchCV.

| Step | Content |
|------|---------|
| 1 | Load Phase 4 artifacts |
| 2 | Data + SMOTE (same setup) |
| 3 | RandomizedSearchCV |
| 4 | Retrain best params on full train set |
| 5 | Before / After comparison |

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])

try:
    import xgboost
except ImportError:
    install("xgboost")

try:
    import imblearn
except ImportError:
    install("imbalanced-learn")

print("Packages ready.")

Packages ready.


In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import time
import warnings
from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, make_scorer
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUTS = Path('/content/drive/MyDrive/nids/outputs')
except ImportError:
    IN_COLAB = False
    OUTPUTS = Path("../outputs")

RESULTS_DIR = OUTPUTS / "results"
MODELS_DIR  = OUTPUTS / "models"
FIGURES_DIR = OUTPUTS / "figures"

RANDOM_STATE = 42
SAMPLE_FRAC  = 0.30
SMOTE_TARGET = 5_000

print(f"OUTPUTS → {OUTPUTS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUTPUTS → /content/drive/MyDrive/nids/outputs


## 1. Load Phase 4 Artifacts

In [3]:
preprocessor = joblib.load(MODELS_DIR / "preprocessor.joblib")
le           = joblib.load(MODELS_DIR / "label_encoder.joblib")

with open(RESULTS_DIR / "kept_features.json") as f:
    kept_features = json.load(f)

with open(RESULTS_DIR / "selected_features.json") as f:
    selected_features = json.load(f)

feat_idx = [kept_features.index(f) for f in selected_features]

print(f"All features     : {len(kept_features)}")
print(f"Selected features: {len(selected_features)}")

All features     : 67
Selected features: 47


## 2. Data + Train/Test Split + SMOTE

In [4]:
df = pd.read_parquet(OUTPUTS / "cicids2017_clean.parquet")
df = df.groupby("Label", group_keys=False).apply(
    lambda g: g.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
)
print(f"Rows: {len(df):,}  (sample_frac={SAMPLE_FRAC})")

feature_cols = [c for c in df.columns if c not in ("Label", "label_enc")]
df["label_enc"] = le.transform(df["Label"])
X = df[feature_cols]
y = df["label_enc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

# Feature selection
X_train_sel = np.ascontiguousarray(X_train_proc[:, feat_idx], dtype=np.float32)
X_test_sel  = np.ascontiguousarray(X_test_proc[:,  feat_idx], dtype=np.float32)
X_train_sel = np.nan_to_num(X_train_sel, nan=0.0, posinf=0.0, neginf=0.0)
X_test_sel  = np.nan_to_num(X_test_sel,  nan=0.0, posinf=0.0, neginf=0.0)

# SMOTE
class_counts      = Counter(y_train.tolist())
majority_class    = max(class_counts, key=class_counts.get)
sampling_strategy = {
    cls: SMOTE_TARGET
    for cls, cnt in class_counts.items()
    if cls != majority_class and cnt < SMOTE_TARGET
}
k_neighbors = min(5, min(class_counts[c] for c in sampling_strategy) - 1)
smote = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=k_neighbors, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_sel, y_train)
print(f"Train after SMOTE: {X_train_smote.shape}")

Rows: 756,239  (sample_frac=0.3)
Train after SMOTE: (651786, 47)


## 3. RandomizedSearchCV

Search space covers the main XGBoost knobs:
- `n_estimators` — more trees → better fit, slower
- `max_depth` — tree complexity
- `learning_rate` — shrinkage per step
- `subsample` / `colsample_bytree` — row/col sampling for variance reduction
- `min_child_weight` — regularisation on leaf splits
- `gamma` — minimum loss reduction to make a split

In [5]:
param_dist = {
    "n_estimators":      [200, 300, 500],
    "max_depth":         [6, 8, 10, 12],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "subsample":         [0.6, 0.8, 1.0],
    "colsample_bytree":  [0.6, 0.8, 1.0],
    "min_child_weight":  [1, 3, 5, 10],
    "gamma":             [0, 0.1, 0.3, 0.5],
}

base_xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=0,
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scorer = make_scorer(f1_score, average="macro", zero_division=0)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring=scorer,
    cv=cv,
    n_jobs=1,        # XGBoost already uses all cores internally
    verbose=2,
    random_state=RANDOM_STATE,
    refit=True,
)

print(f"Search space: {30} iterations × 3-fold CV = 90 fits")
print("Starting search...")
t0 = time.time()
search.fit(X_train_smote, y_train_smote)
print(f"\nSearch complete in {(time.time()-t0)/60:.1f} min")
print(f"Best CV F1 Macro : {search.best_score_:.4f}")
print(f"Best params      : {search.best_params_}")

Search space: 30 iterations × 3-fold CV = 90 fits
Starting search...
Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END colsample_bytree=0.6, gamma=0.1, learning_rate=0.05, max_depth=12, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  26.5s
[CV] END colsample_bytree=0.6, gamma=0.1, learning_rate=0.05, max_depth=12, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  25.9s
[CV] END colsample_bytree=0.6, gamma=0.1, learning_rate=0.05, max_depth=12, min_child_weight=10, n_estimators=300, subsample=1.0; total time=  26.3s
[CV] END colsample_bytree=1.0, gamma=0.1, learning_rate=0.05, max_depth=8, min_child_weight=5, n_estimators=500, subsample=1.0; total time=  35.4s
[CV] END colsample_bytree=1.0, gamma=0.1, learning_rate=0.05, max_depth=8, min_child_weight=5, n_estimators=500, subsample=1.0; total time=  36.2s
[CV] END colsample_bytree=1.0, gamma=0.1, learning_rate=0.05, max_depth=8, min_child_weight=5, n_estimators=500, subsample=1.0; t

## 4. Retrain Best Model + Evaluate

In [6]:
best_params = search.best_params_

xgb_tuned = XGBClassifier(
    **best_params,
    use_label_encoder=False,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=0,
)

t0 = time.time()
xgb_tuned.fit(X_train_smote, y_train_smote)
print(f"Training time: {(time.time()-t0)/60:.1f} min")

joblib.dump(xgb_tuned, MODELS_DIR / "xgboost_tuned.joblib")
print(f"Saved → {MODELS_DIR / 'xgboost_tuned.joblib'}")

Training time: 0.4 min
Saved → /content/drive/MyDrive/nids/outputs/models/xgboost_tuned.joblib


## 5. Before / After Comparison

In [7]:
xgb_phase4 = joblib.load(MODELS_DIR / "xgboost_feat_eng.joblib")

def get_metrics(model, X, y_true):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)
    return {
        "accuracy":    accuracy_score(y_true, y_pred),
        "f1_macro":    f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc":     roc_auc_score(y_true, y_prob, multi_class="ovr", average="weighted"),
    }

before = get_metrics(xgb_phase4, X_test_sel, y_test)
after  = get_metrics(xgb_tuned,  X_test_sel, y_test)

print("\n" + "═"*54)
print(f"{'Metric':<18} {'Phase 4':>10} {'Phase 5':>10} {'Delta':>10}")
print("─"*54)
for k in ["f1_macro", "f1_weighted", "roc_auc", "accuracy"]:
    d = after[k] - before[k]
    print(f"{k:<18} {before[k]:>10.4f} {after[k]:>10.4f} {d:>+10.4f}")
print("═"*54)

print("\n─── Classification Report (Phase 5) ───")
y_pred = xgb_tuned.predict(X_test_sel)
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))


══════════════════════════════════════════════════════
Metric                Phase 4    Phase 5      Delta
──────────────────────────────────────────────────────
f1_macro               0.8605     0.7871    -0.0734
f1_weighted            0.9987     0.9986    -0.0001
roc_auc                1.0000     1.0000    -0.0000
accuracy               0.9987     0.9986    -0.0001
══════════════════════════════════════════════════════

─── Classification Report (Phase 5) ───
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    125703
                       Bot       0.75      0.91      0.83       117
                      DDoS       1.00      1.00      1.00      7681
             DoS GoldenEye       1.00      1.00      1.00       617
                  DoS Hulk       1.00      1.00      1.00     10371
          DoS Slowhttptest       0.98      0.99      0.99       314
             DoS slowloris       0.99      0.99      0.9

In [8]:
# ── Save results ──────────────────────────────────────────────────────
rows = [
    {"model": "XGBoost Phase4 (selected features)", **before},
    {"model": "XGBoost Phase5 (tuned)",             **after},
]
comp_df = pd.DataFrame(rows)
comp_df.to_csv(RESULTS_DIR / "hyperopt_comparison.csv", index=False)
print(f"Saved → {RESULTS_DIR / 'hyperopt_comparison.csv'}")

cv_df = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cv_df.to_csv(RESULTS_DIR / "hyperopt_cv_results.csv", index=False)
print(f"Saved → {RESULTS_DIR / 'hyperopt_cv_results.csv'}")

with open(RESULTS_DIR / "best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
print(f"Saved → {RESULTS_DIR / 'best_params.json'}")
print(comp_df.to_string(index=False))

Saved → /content/drive/MyDrive/nids/outputs/results/hyperopt_comparison.csv
Saved → /content/drive/MyDrive/nids/outputs/results/hyperopt_cv_results.csv
Saved → /content/drive/MyDrive/nids/outputs/results/best_params.json
                             model  accuracy  f1_macro  f1_weighted  roc_auc
XGBoost Phase4 (selected features)  0.998671  0.860473     0.998683 0.999977
            XGBoost Phase5 (tuned)  0.998585  0.787118     0.998584 0.999954
